# Preparation

In [1]:
#@title Import Libs
#@markdown All libraries import goes over here

import os
import glob
import joblib
import subprocess

In [2]:
#@title Import Datasets

USE_MY_DRIVE = True #@param {type:"boolean"}
#@markdown > If you decided to **USE_MY_DRIVE** make sure to first run the latest version of [**02_data_preparation.ipynb**](https://colab.research.google.com/drive/1mdcqA8y9RN-pgtv3hRnDm2ZsheJqQnSe?usp=drive_link) with save to drive on
SHARED_FOLDER_URL = 'https://drive.google.com/drive/folders/1oj1cqdfyh1ouXThXJbjgamLA6H5nX8nF'

data_dir = 'processed_data'
loaded_from_my_drive = False

if USE_MY_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_path = '/content/drive/MyDrive/DataScience-ProjectUAS/processed_data'
        if os.path.exists(f'{drive_path}/dataset_1_splits.pkl'):
            data_dir = drive_path
            loaded_from_my_drive = True
            print(f"Loaded from personal Google Drive: {data_dir}")
    except ImportError:
        pass

if not loaded_from_my_drive:
    os.makedirs(data_dir, exist_ok=True)
    if not glob.glob(f'{data_dir}/**/dataset_1_splits.pkl', recursive=True):
        print("Downloading splits from shared Google Drive folder...")
        subprocess.run(
            ['gdown', '--folder', SHARED_FOLDER_URL, '-O', data_dir, '--remaining-ok'],
            check=True
        )
    matched = glob.glob(f'{data_dir}/**/dataset_1_splits.pkl', recursive=True)
    data_dir = os.path.dirname(matched[0])
    print(f"Loaded from shared Drive fallback: {data_dir}")

dataset_1 = joblib.load(f'{data_dir}/dataset_1_splits.pkl')
dataset_2 = joblib.load(f'{data_dir}/dataset_2_splits.pkl')
dataset_3 = joblib.load(f'{data_dir}/dataset_3_splits.pkl')

print("All datasets ready for modeling.")

Mounted at /content/drive
Loaded from shared Drive fallback: processed_data
All datasets ready for modeling.


Import Library for Modeling

In [3]:
import time
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

print("Library Baseline siap digunakan.")

Library Baseline siap digunakan.


Konfigurasi Model Baseline

In [4]:
# Model 1: Linear Regression

datasets_splits = {
    'Dataset 1': dataset_1,
    'Dataset 2': dataset_2,
    'Dataset 3': dataset_3
}

baseline_params = {
    'fit_intercept': True
}

print("Model: Linear Regression")
print("Hyperparameter:", baseline_params)
print("Jumlah dataset:", len(datasets_splits))

Model: Linear Regression
Hyperparameter: {'fit_intercept': True}
Jumlah dataset: 3


Training Linear Regression

In [5]:
baseline_models = {}
training_times = {}

for nama, split in datasets_splits.items():

    start_time = time.time()

    model = LinearRegression(**baseline_params)

    model.fit(
        split['X_train'],
        split['y_train']
    )

    elapsed = time.time() - start_time

    baseline_models[nama] = model
    training_times[nama] = round(elapsed, 2)

    print(f"{nama}: Training selesai ({elapsed:.2f} detik)")

print("\nSeluruh model Baseline berhasil dilatih.")

Dataset 1: Training selesai (1.89 detik)
Dataset 2: Training selesai (1.55 detik)
Dataset 3: Training selesai (0.01 detik)

Seluruh model Baseline berhasil dilatih.


Evaluasi Training & Validation

In [6]:
baseline_results = []

for nama, split in datasets_splits.items():

    model = baseline_models[nama]

    # Prediksi data training
    y_train_pred = model.predict(split['X_train'])

    # Prediksi data validation
    y_val_pred = model.predict(split['X_val'])

    # Menghitung evaluasi
    train_rmse = np.sqrt(
        mean_squared_error(
            split['y_train'],
            y_train_pred
        )
    )

    val_rmse = np.sqrt(
        mean_squared_error(
            split['y_val'],
            y_val_pred
        )
    )

    val_mae = mean_absolute_error(
        split['y_val'],
        y_val_pred
    )

    val_r2 = r2_score(
        split['y_val'],
        y_val_pred
    )

    baseline_results.append({
        'Dataset': nama,
        'Train_RMSE': train_rmse,
        'Val_RMSE': val_rmse,
        'Val_MAE': val_mae,
        'Val_R2': val_r2,
        'Train_Time_Sec': training_times[nama]
    })

# Membuat tabel hasil evaluasi
baseline_metrics_df = pd.DataFrame(baseline_results)

display(baseline_metrics_df.round(4))

,Dataset,Train_RMSE,Val_RMSE,Val_MAE,Val_R2,Train_Time_Sec
0,Dataset 1,41750.6430,43524.4545,30270.2684,0.7472,1.89
1,Dataset 2,0.5001,0.4999,0.3989,0.9130,1.55
2,Dataset 3,5.0015,5.0935,4.0599,0.9822,0.01


Simpan Model & Hasil Evaluasi

In [7]:
model_dir = 'models'

if os.path.exists('/content/drive/MyDrive/DataScience-ProjectUAS'):
    model_dir = '/content/drive/MyDrive/DataScience-ProjectUAS/models'

os.makedirs(model_dir, exist_ok=True)

baseline_metrics_df.to_csv(
    f'{model_dir}/baseline_linear_regression_val_metrics.csv',
    index=False
)


for nama, model in baseline_models.items():

    filename = nama.lower().replace(' ', '_')

    joblib.dump(
        model,
        f'{model_dir}/{filename}_linear_regression.pkl'
    )

print("Model dan hasil evaluasi berhasil disimpan.")

Model dan hasil evaluasi berhasil disimpan.
